In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/mahmudurrahmansupto/baangla-btrc/combined_dataset_v3-002.jsonl


In [2]:
print("hello")

hello


In [ ]:
print ("name is mahmudur rahman supto")

In [1]:
print ("hello world")

hello world


In [ ]:
import glob, json, re, collections
import pandas as pd
import numpy as np

PATH = glob.glob('/kaggle/input/**/combined_dataset_v3-002.jsonl', recursive=True)[0]
print(PATH)

rows, bad = [], 0
with open(PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            bad += 1
print(f"records: {len(rows):,} | malformed lines: {bad}")
df = pd.json_normalize(rows)
print(df.shape)
df.head()

In [ ]:
# Schema: dtypes, nulls, example values
info = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isna().sum(),
    'null_%': (df.isna().mean()*100).round(2),
    'unique': df.apply(lambda s: s.astype(str).nunique()),
    'example': df.apply(lambda s: str(s.dropna().iloc[0])[:80] if s.notna().any() else None),
})
info

In [ ]:
# Identify text columns and compute length stats (chars / words)
text_cols = [c for c in df.columns if df[c].map(lambda x: isinstance(x, str)).mean() > 0.9
             and df[c].dropna().astype(str).str.len().mean() > 20]
print("text columns:", text_cols)

stats = {}
for c in text_cols:
    s = df[c].fillna('').astype(str)
    stats[c] = {
        'empty': int((s.str.strip()=='').sum()),
        'chars_mean': s.str.len().mean(), 'chars_median': s.str.len().median(),
        'chars_p95': s.str.len().quantile(.95), 'chars_max': s.str.len().max(),
        'words_mean': s.str.split().str.len().mean(),
        'words_p95': s.str.split().str.len().quantile(.95),
    }
pd.DataFrame(stats).T.round(1)

In [ ]:
# Duplicates and low-cardinality (label/category) columns
print("exact duplicate rows:", df.astype(str).duplicated().sum())
for c in text_cols:
    print(f"duplicate '{c}':", df[c].duplicated().sum())

for c in df.columns:
    if c in text_cols: continue
    n = df[c].astype(str).nunique()
    if n <= 30:
        print(f"\n== {c} ({n} values) ==")
        print(df[c].astype(str).value_counts().head(30))

In [ ]:
# Script composition: Bangla vs Latin vs digits vs other, per text column
def script_ratio(t):
    t = str(t); n = max(len(t), 1)
    bn = len(re.findall(r'[\u0980-\u09FF]', t))
    en = len(re.findall(r'[A-Za-z]', t))
    dg = len(re.findall(r'[0-9\u09E6-\u09EF]', t))
    return bn/n, en/n, dg/n

for c in text_cols:
    r = np.array([script_ratio(t) for t in df[c].fillna('').head(50000)])
    print(f"{c}: bangla={r[:,0].mean():.2%} latin={r[:,1].mean():.2%} digits={r[:,2].mean():.2%}")
    print("  rows with <50% Bangla chars:", int((r[:,0] < .5).sum()))

In [ ]:
# Data quality: URLs, HTML, control/odd chars, leading/trailing whitespace
checks = {
    'url': r'https?://|www\.',
    'html_tag': r'<[^>]+>',
    'zero_width': r'[\u200b\u200c\u200d\ufeff]',
    'repeated_chars(5+)': r'(.)\1{4,}',
    'multi_space': r'\s{3,}',
}
q = {}
for c in text_cols:
    s = df[c].fillna('').astype(str)
    q[c] = {k: int(s.str.contains(v, regex=True).sum()) for k, v in checks.items()}
    q[c]['edge_whitespace'] = int((s != s.str.strip()).sum())
pd.DataFrame(q).T

In [ ]:
# Character & token frequency (Bangla vocabulary overview)
main = max(text_cols, key=lambda c: df[c].astype(str).str.len().sum())
print("main text column:", main)
sample = df[main].dropna().astype(str).sample(min(50000, len(df)), random_state=0)
words = collections.Counter(w for t in sample for w in re.findall(r'[\u0980-\u09FF]+', t))
print("vocab size (sample):", len(words), "| total tokens:", sum(words.values()))
pd.DataFrame(words.most_common(30), columns=['word','count'])

In [ ]:
# Distribution plots
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(text_cols), figsize=(6*len(text_cols), 4), squeeze=False)
for ax, c in zip(axes[0], text_cols):
    df[c].fillna('').astype(str).str.split().str.len().clip(upper=df[c].fillna('').astype(str).str.split().str.len().quantile(.99)).hist(bins=50, ax=ax)
    ax.set_title(f'{c}: words per record'); ax.set_xlabel('words'); ax.set_ylabel('records')
plt.tight_layout(); plt.show()

In [ ]:
# Random samples for manual inspection
for _, r in df.sample(5, random_state=1).iterrows():
    print(json.dumps(r.to_dict(), ensure_ascii=False, default=str)[:600]); print('-'*60)